# Stocker Trader Bot - Backtesting

Backtest the RSI strategy on historical data for stocks and crypto.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Optional
import matplotlib.pyplot as plt

from src.broker.alpaca_client import get_alpaca_client
from config.settings import settings

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

client = get_alpaca_client()
print('Alpaca client connected')

## 1. Fetch Historical Data

In [ ]:
def fetch_historical_data(symbol: str, days: int = 365, timeframe: str = '1Day') -> pd.DataFrame:
    """Fetch historical OHLCV data for a symbol."""
    start = datetime.now() - timedelta(days=days)
    end = datetime.now()
    
    bars = client.get_bars(symbol, timeframe, start=start, end=end, limit=days)
    
    if hasattr(bars, 'df'):
        df = bars.df.reset_index()
        df.columns = [c.lower() for c in df.columns]
        if 'timestamp' in df.columns:
            df = df.rename(columns={'timestamp': 'date'})
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
        return df
    return pd.DataFrame()

# Test with SPY
spy_data = fetch_historical_data('SPY', days=365)
print(f'SPY: {len(spy_data)} bars')
spy_data.tail()

In [ ]:
# Fetch data for backtest symbols
BACKTEST_SYMBOLS = ['SPY', 'AAPL', 'MSFT', 'NVDA', 'AMD']
CRYPTO_SYMBOLS = ['BTC/USD', 'ETH/USD']

data = {}
for symbol in BACKTEST_SYMBOLS:
    try:
        df = fetch_historical_data(symbol, days=365)
        if len(df) > 0:
            data[symbol] = df
            print(f'{symbol}: {len(df)} bars')
    except Exception as e:
        print(f'{symbol}: Error - {e}')

for symbol in CRYPTO_SYMBOLS:
    try:
        df = fetch_historical_data(symbol, days=365)
        if len(df) > 0:
            data[symbol] = df
            print(f'{symbol}: {len(df)} bars')
    except Exception as e:
        print(f'{symbol}: Error - {e}')

## 2. Technical Indicators

In [ ]:
def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    """Calculate RSI indicator."""
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_ema(prices: pd.Series, period: int = 20) -> pd.Series:
    """Calculate Exponential Moving Average."""
    return prices.ewm(span=period, adjust=False).mean()

def calculate_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Calculate Average True Range."""
    high = df['high']
    low = df['low']
    close = df['close']
    
    tr1 = high - low
    tr2 = abs(high - close.shift())
    tr3 = abs(low - close.shift())
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(window=period).mean()
    return atr

def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Add all technical indicators to dataframe."""
    df = df.copy()
    df['rsi'] = calculate_rsi(df['close'], 14)
    df['ema_20'] = calculate_ema(df['close'], 20)
    df['ema_50'] = calculate_ema(df['close'], 50)
    df['atr'] = calculate_atr(df, 14)
    df['atr_pct'] = df['atr'] / df['close'] * 100
    return df

# Add indicators to all data
for symbol in data:
    data[symbol] = add_indicators(data[symbol])

data['SPY'][['close', 'rsi', 'ema_20', 'atr_pct']].tail(10)

## 3. Backtest Engine

In [ ]:
@dataclass
class Trade:
    symbol: str
    direction: str
    entry_date: datetime
    entry_price: float
    exit_date: Optional[datetime] = None
    exit_price: Optional[float] = None
    shares: float = 0
    pnl: float = 0
    pnl_pct: float = 0
    exit_reason: str = ''

@dataclass 
class BacktestConfig:
    initial_capital: float = 10000
    risk_per_trade: float = 0.02  # 2%
    max_position_size: float = 0.10  # 10%
    stop_loss_pct: float = 0.02  # 2%
    take_profit_pct: float = 0.04  # 4%
    rsi_oversold: int = 30
    rsi_overbought: int = 70
    require_trend: bool = True  # EMA trend filter

class Backtester:
    def __init__(self, config: BacktestConfig):
        self.config = config
        self.trades: list[Trade] = []
        self.equity_curve: list[float] = []
        
    def run(self, df: pd.DataFrame, symbol: str) -> dict:
        """Run backtest on a single symbol."""
        capital = self.config.initial_capital
        position = None
        self.equity_curve = [capital]
        
        # Use crypto settings if applicable
        is_crypto = symbol in CRYPTO_SYMBOLS
        stop_loss_pct = 0.04 if is_crypto else self.config.stop_loss_pct
        take_profit_pct = 0.08 if is_crypto else self.config.take_profit_pct
        max_position = 0.05 if is_crypto else self.config.max_position_size
        
        for i in range(50, len(df)):  # Start after warmup period
            row = df.iloc[i]
            prev_row = df.iloc[i-1]
            date = df.index[i]
            price = row['close']
            
            # Check for exit if in position
            if position:
                # Calculate current P&L
                if position.direction == 'long':
                    current_pnl_pct = (price - position.entry_price) / position.entry_price
                else:
                    current_pnl_pct = (position.entry_price - price) / position.entry_price
                
                exit_reason = None
                
                # Stop loss
                if current_pnl_pct <= -stop_loss_pct:
                    exit_reason = 'stop_loss'
                # Take profit
                elif current_pnl_pct >= take_profit_pct:
                    exit_reason = 'take_profit'
                # RSI reversal exit
                elif position.direction == 'long' and row['rsi'] > self.config.rsi_overbought:
                    exit_reason = 'rsi_overbought'
                elif position.direction == 'short' and row['rsi'] < self.config.rsi_oversold:
                    exit_reason = 'rsi_oversold'
                
                if exit_reason:
                    # Close position
                    pnl = position.shares * (price - position.entry_price)
                    if position.direction == 'short':
                        pnl = -pnl
                    
                    position.exit_date = date
                    position.exit_price = price
                    position.pnl = pnl
                    position.pnl_pct = current_pnl_pct * 100
                    position.exit_reason = exit_reason
                    
                    capital += pnl
                    self.trades.append(position)
                    position = None
            
            # Check for entry if no position
            if position is None:
                signal = None
                
                # Long signal: RSI oversold + price above EMA (uptrend)
                if row['rsi'] < self.config.rsi_oversold:
                    if not self.config.require_trend or price > row['ema_20']:
                        signal = 'long'
                
                # Short signal: RSI overbought + price below EMA (downtrend)
                elif row['rsi'] > self.config.rsi_overbought:
                    if not self.config.require_trend or price < row['ema_20']:
                        signal = 'short'
                
                if signal:
                    # Position sizing
                    risk_amount = capital * self.config.risk_per_trade
                    risk_per_share = price * stop_loss_pct
                    shares_by_risk = risk_amount / risk_per_share
                    
                    max_position_value = capital * max_position
                    shares_by_position = max_position_value / price
                    
                    shares = min(shares_by_risk, shares_by_position)
                    if is_crypto:
                        shares = round(shares, 6)
                    else:
                        shares = int(shares)
                    
                    if shares > 0:
                        position = Trade(
                            symbol=symbol,
                            direction=signal,
                            entry_date=date,
                            entry_price=price,
                            shares=shares
                        )
            
            # Track equity
            if position:
                unrealized = position.shares * (price - position.entry_price)
                if position.direction == 'short':
                    unrealized = -unrealized
                self.equity_curve.append(capital + unrealized)
            else:
                self.equity_curve.append(capital)
        
        # Close any open position at end
        if position:
            price = df.iloc[-1]['close']
            pnl = position.shares * (price - position.entry_price)
            if position.direction == 'short':
                pnl = -pnl
            position.exit_date = df.index[-1]
            position.exit_price = price
            position.pnl = pnl
            position.pnl_pct = ((price - position.entry_price) / position.entry_price) * 100
            position.exit_reason = 'end_of_data'
            capital += pnl
            self.trades.append(position)
        
        return self._calculate_metrics(symbol)
    
    def _calculate_metrics(self, symbol: str) -> dict:
        """Calculate backtest performance metrics."""
        if not self.trades:
            return {'symbol': symbol, 'total_trades': 0}
        
        trades_df = pd.DataFrame([vars(t) for t in self.trades])
        
        total_trades = len(trades_df)
        winning_trades = len(trades_df[trades_df['pnl'] > 0])
        losing_trades = len(trades_df[trades_df['pnl'] < 0])
        
        win_rate = winning_trades / total_trades if total_trades > 0 else 0
        
        total_pnl = trades_df['pnl'].sum()
        avg_win = trades_df[trades_df['pnl'] > 0]['pnl'].mean() if winning_trades > 0 else 0
        avg_loss = trades_df[trades_df['pnl'] < 0]['pnl'].mean() if losing_trades > 0 else 0
        
        # Profit factor
        gross_profit = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
        gross_loss = abs(trades_df[trades_df['pnl'] < 0]['pnl'].sum())
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
        
        # Max drawdown from equity curve
        equity = pd.Series(self.equity_curve)
        peak = equity.expanding().max()
        drawdown = (equity - peak) / peak
        max_drawdown = drawdown.min()
        
        # Return
        total_return = (self.equity_curve[-1] - self.config.initial_capital) / self.config.initial_capital
        
        # Sharpe ratio (simplified, assuming 252 trading days)
        returns = equity.pct_change().dropna()
        sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
        
        return {
            'symbol': symbol,
            'total_trades': total_trades,
            'winning_trades': winning_trades,
            'losing_trades': losing_trades,
            'win_rate': win_rate,
            'total_pnl': total_pnl,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'profit_factor': profit_factor,
            'max_drawdown': max_drawdown,
            'total_return': total_return,
            'sharpe_ratio': sharpe,
            'final_equity': self.equity_curve[-1]
        }

print('Backtester ready')

## 4. Run Backtest

In [ ]:
# Run backtest on all symbols
config = BacktestConfig(
    initial_capital=10000,
    risk_per_trade=0.02,
    stop_loss_pct=0.02,
    take_profit_pct=0.04,
    rsi_oversold=30,
    rsi_overbought=70,
    require_trend=True
)

results = []
all_trades = []
equity_curves = {}

for symbol, df in data.items():
    if len(df) < 100:
        print(f'{symbol}: Not enough data')
        continue
    
    bt = Backtester(config)
    metrics = bt.run(df, symbol)
    results.append(metrics)
    all_trades.extend(bt.trades)
    equity_curves[symbol] = bt.equity_curve
    
    print(f"{symbol}: {metrics['total_trades']} trades, "
          f"Win rate: {metrics['win_rate']:.1%}, "
          f"Return: {metrics['total_return']:.1%}, "
          f"Max DD: {metrics['max_drawdown']:.1%}")

results_df = pd.DataFrame(results)
results_df

## 5. Visualizations

In [ ]:
# Plot equity curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Equity curves
ax = axes[0, 0]
for symbol, equity in equity_curves.items():
    ax.plot(equity, label=symbol, alpha=0.8)
ax.axhline(y=config.initial_capital, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Equity Curves')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Portfolio Value ($)')
ax.legend(loc='upper left')

# Win rate by symbol
ax = axes[0, 1]
colors = ['green' if r > 0.5 else 'red' for r in results_df['win_rate']]
ax.bar(results_df['symbol'], results_df['win_rate'] * 100, color=colors, alpha=0.7)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Win Rate by Symbol')
ax.set_ylabel('Win Rate (%)')
ax.tick_params(axis='x', rotation=45)

# Total return by symbol
ax = axes[1, 0]
colors = ['green' if r > 0 else 'red' for r in results_df['total_return']]
ax.bar(results_df['symbol'], results_df['total_return'] * 100, color=colors, alpha=0.7)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Total Return by Symbol')
ax.set_ylabel('Return (%)')
ax.tick_params(axis='x', rotation=45)

# Profit factor
ax = axes[1, 1]
pf = results_df['profit_factor'].replace([np.inf], 5).clip(upper=5)
colors = ['green' if p > 1 else 'red' for p in pf]
ax.bar(results_df['symbol'], pf, color=colors, alpha=0.7)
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Profit Factor by Symbol')
ax.set_ylabel('Profit Factor')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Trade analysis
if all_trades:
    trades_df = pd.DataFrame([vars(t) for t in all_trades])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # P&L distribution
    ax = axes[0]
    trades_df['pnl'].hist(bins=30, ax=ax, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(x=0, color='red', linestyle='--')
    ax.axvline(x=trades_df['pnl'].mean(), color='green', linestyle='--', label=f"Mean: ${trades_df['pnl'].mean():.2f}")
    ax.set_title('P&L Distribution')
    ax.set_xlabel('P&L ($)')
    ax.set_ylabel('Frequency')
    ax.legend()
    
    # Exit reasons
    ax = axes[1]
    exit_counts = trades_df['exit_reason'].value_counts()
    ax.pie(exit_counts, labels=exit_counts.index, autopct='%1.1f%%', startangle=90)
    ax.set_title('Exit Reasons')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal Trades: {len(trades_df)}")
    print(f"Average P&L: ${trades_df['pnl'].mean():.2f}")
    print(f"Median P&L: ${trades_df['pnl'].median():.2f}")
    print(f"Std Dev: ${trades_df['pnl'].std():.2f}")

## 6. Parameter Optimization

In [ ]:
# Test different RSI thresholds
def optimize_rsi_thresholds(df: pd.DataFrame, symbol: str):
    """Find optimal RSI thresholds."""
    results = []
    
    for oversold in [20, 25, 30, 35]:
        for overbought in [65, 70, 75, 80]:
            config = BacktestConfig(
                initial_capital=10000,
                rsi_oversold=oversold,
                rsi_overbought=overbought,
                require_trend=True
            )
            bt = Backtester(config)
            metrics = bt.run(df.copy(), symbol)
            metrics['rsi_oversold'] = oversold
            metrics['rsi_overbought'] = overbought
            results.append(metrics)
    
    return pd.DataFrame(results)

# Optimize on SPY
if 'SPY' in data:
    opt_results = optimize_rsi_thresholds(data['SPY'], 'SPY')
    opt_results = opt_results.sort_values('total_return', ascending=False)
    print("Top 5 RSI configurations for SPY:")
    opt_results[['rsi_oversold', 'rsi_overbought', 'total_trades', 'win_rate', 'total_return', 'sharpe_ratio']].head()

In [ ]:
# Heatmap of returns by RSI thresholds
if 'SPY' in data:
    pivot = opt_results.pivot(index='rsi_oversold', columns='rsi_overbought', values='total_return')
    
    plt.figure(figsize=(8, 6))
    plt.imshow(pivot.values * 100, cmap='RdYlGn', aspect='auto')
    plt.colorbar(label='Return (%)')
    plt.xticks(range(len(pivot.columns)), pivot.columns)
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xlabel('RSI Overbought')
    plt.ylabel('RSI Oversold')
    plt.title('SPY Returns by RSI Thresholds')
    
    # Add text annotations
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            plt.text(j, i, f'{pivot.values[i, j]*100:.1f}%', 
                     ha='center', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 7. Summary

In [ ]:
# Overall summary
print("="*60)
print("BACKTEST SUMMARY")
print("="*60)
print(f"\nPeriod: {data['SPY'].index[0].strftime('%Y-%m-%d')} to {data['SPY'].index[-1].strftime('%Y-%m-%d')}")
print(f"Initial Capital: ${config.initial_capital:,.2f}")
print(f"Risk per Trade: {config.risk_per_trade:.1%}")
print(f"Stop Loss: {config.stop_loss_pct:.1%} (stocks), 4% (crypto)")
print(f"Take Profit: {config.take_profit_pct:.1%} (stocks), 8% (crypto)")

print(f"\n{'Symbol':<12} {'Trades':<8} {'Win Rate':<10} {'Return':<10} {'Max DD':<10} {'Sharpe':<8}")
print("-"*60)
for _, row in results_df.iterrows():
    print(f"{row['symbol']:<12} {row['total_trades']:<8} {row['win_rate']:.1%}     {row['total_return']:>+.1%}     {row['max_drawdown']:.1%}     {row['sharpe_ratio']:.2f}")

# Aggregate stats
print("\n" + "="*60)
print("AGGREGATE STATISTICS")
print("="*60)
print(f"Total Trades: {results_df['total_trades'].sum()}")
print(f"Avg Win Rate: {results_df['win_rate'].mean():.1%}")
print(f"Avg Return: {results_df['total_return'].mean():.1%}")
print(f"Best Performer: {results_df.loc[results_df['total_return'].idxmax(), 'symbol']} ({results_df['total_return'].max():.1%})")
print(f"Worst Performer: {results_df.loc[results_df['total_return'].idxmin(), 'symbol']} ({results_df['total_return'].min():.1%})")